# Demo · registering a post-Xenium H&E in code (10x human liver cancer) · from the raw data
The live demo notebook loads its results from `liver_prepared/`. **This version needs nothing precomputed:** part 0 runs
the whole-section registration (three attempts, each scored against the 10x keypoint alignment), cuts the 1 mm crop and
draws the figures, from the 10x files in `data/10x_Liver_cancer/`, and writes them to `results_from_raw/liver/`.
Part 0 takes **about 7 min and ~6 GB of RAM**; re-running skips it when its outputs exist.

**Run it from the `workshop/` folder** (next to `data/` and `handson/`), in the `merit` environment. Needed from the
10x download: the `…_outs/` folder (with `transcripts.parquet` and `cells.zarr.zip`), `…_he_image.ome.tif` and
`…_he_imagealignment.csv`.

In [ ]:
import json, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import spatialdata as sd, spatialdata_plot  # noqa: F401
import multispace as ms
from spatialdata.transformations import get_transformation, set_transformation, Identity
from IPython.display import Image, display
import sys
sys.path.insert(0, "handson/liver")             # prepare_liver.py, liver_lib.py, make_liver_figures.py
DATA = Path("data/10x_Liver_cancer")             # 10x download: …_outs/, …_he_image.ome.tif, …_he_imagealignment.csv
PREP = Path("results_from_raw/liver")            # everything part 0 computes

## 0 · Rebuild from the raw data (~7 min the first time)
Same functions as `handson/liver/prepare_liver.py` (open it to see each step in full).

**0.1 · Whole-section registration.** multispace reads the run (`read_st`) and adds the H&E at pyramid level 1
(0.55 µm/px). Three attempts, each scored by the H&E–DAPI correlation and by the distance to the 10x keypoint alignment:
`ms.register_image` defaults; `pre_rotation="auto"`; and the latter refined on tiles (phase correlation, RANSAC affine).

In [ ]:
import gc
import prepare_liver as pl
PREP.mkdir(parents=True, exist_ok=True)
if not (PREP / "M_refined.npy").exists():
    s, res, M1 = pl.register(DATA, PREP)
    del s; gc.collect()

**0.2 · The 1 mm crop** (tumour / hepatocyte border) with every layer: DAPI, the H&E pixels under it with the refined
transform, cells and the transcripts read from `transcripts.parquet` with a filter (loading all 194 million would not fit).

In [ ]:
if not (PREP / "liver_crop.zarr").exists():
    crop = pl.save_crop(DATA, PREP)
    json.dump({"crop_xenium_px": crop}, open(PREP / "crop.json", "w"))
    gc.collect()

**0.3 · Figures L1–L4** (the same as in the slides).

In [ ]:
%run handson/liver/make_liver_figures.py --data {DATA} --out {PREP}
%matplotlib inline

## 1 · Whole section: three attempts, scored

In [ ]:
reg = json.load(open(PREP / "registration.json"))
pd.DataFrame({k: {c: v[c] for c in ("median_um_vs_10x", "max_um_vs_10x", "r_he_dapi")} for k, v in reg.items()}).T
# r_he_dapi: correlation of H&E haematoxylin (colour deconvolution) with DAPI after warping: nuclei overlap -> high

In [ ]:
display(Image(PREP / "figures" / "L2_registration_attempts.png", width=1100))
display(Image(PREP / "figures" / "L3_refinement_shift_field.png", width=800))

**Why the default fails.** The H&E was scanned rotated by 90°. Intensity-based optimisation only searches near its
starting point, so it converges to a wrong local optimum and still returns a matrix: nothing crashes. `pre_rotation="auto"`
tries the four orientations first. The remaining ~1–2% scale error (L3) is then measured on tiles and removed with one affine.

## 2 · The 1 mm crop: one object, every layer

In [ ]:
c = sd.read_zarr(PREP / "liver_crop.zarr")
c

In [ ]:
x0, y0, size = json.load(open(PREP / "crop.json"))["crop_xenium_px"]
w = 600; bx, by = x0 + size // 2 - w // 2, y0 + size // 2 - w // 2       # 130 µm window
win = sd.bounding_box_query(c, axes=("x", "y"), min_coordinate=[bx, by], max_coordinate=[bx + w, by + w],
                            target_coordinate_system="global")
fig, axs = plt.subplots(1, 2, figsize=(13, 6.5))
(win.pl.render_images("he").pl.render_shapes("cell_boundaries", fill_alpha=0, outline_alpha=0.8, outline_color="black")
    .pl.show(coordinate_systems="global", ax=axs[0], title="H&E (registered) + Xenium cells"))
(win.pl.render_images("morphology_focus", cmap="gray")
    .pl.render_points("transcripts", color="feature_name", groups=["GPC3", "CYP3A4", "PTPRC", "PECAM1"],
                      palette=["red", "lime", "deepskyblue", "gold"], size=3)
    .pl.show(coordinate_systems="global", ax=axs[1], title="DAPI + transcripts"))

## 3 · Live: register the crop with multispace
The crop's H&E is reset to its own pixel frame, then registered onto the DAPI crop. Compare with the transform from the
whole-section run (refined), which the crop came with.

In [ ]:
M_ref = get_transformation(c.images["he"], to_coordinate_system="global").to_affine_matrix(("x", "y"), ("x", "y"))
live = sd.SpatialData(images={"morphology_focus": c.images["morphology_focus"], "he": c.images["he"]})
set_transformation(live.images["he"], {"he": Identity()}, set_all=True)
live.attrs["multispace"] = {"images": {"he": {"modality": "he"}}}
res = ms.register_image(live, "he", "morphology_focus", method="affine", pre_rotation="auto", max_side=1500)
M_live = get_transformation(live.images["he"], to_coordinate_system="global").to_affine_matrix(("x", "y"), ("x", "y"))
h, w_ = c.images["he"].shape[1:]
P = np.array([[0, 0, 1], [w_, 0, 1], [0, h, 1], [w_, h, 1], [w_ / 2, h / 2, 1]]).T
d_um = np.linalg.norm((M_live @ P - M_ref @ P)[:2], axis=0) * 0.2125
print("live crop registration vs whole-section refined transform: median %.1f µm, max %.1f µm" % (np.median(d_um), d_um.max()))

**Questions.** How would you know the registration is right if there were no 10x keypoints? (Score it: nuclei
overlap, a shift scan, a few trusted landmarks.) Why is a small crop easier to register than the whole section? What
changes when the target is not DAPI but a protein image (UnumLocalia, next)?